# Fraud / Anomaly Detection
## Notebook 08 — Local Outlier Factor

### Objective
Detect anomalies that are sparse relative to their local neighborhoods.

## Imports and sample

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from src.fraud_detection import stratified_working_sample
from sklearn.neighbors import LocalOutlierFactor
Xw, yw = stratified_working_sample(X, y, max_rows=30000, random_state=RANDOM_STATE)

**Interpretation.** LOF is quadratic-ish in practical neighbor search cost, so a smaller sample is intentional.

**ML decision.** Use novelty mode only when scoring unseen data; this experiment evaluates the fitted sample.

## Fit and evaluate

In [ ]:
model = LocalOutlierFactor(n_neighbors=35, contamination='auto', n_jobs=-1)
model.fit_predict(Xw)
scores = -model.negative_outlier_factor_
result = evaluate_scores(yw, scores, 'Local Outlier Factor')
pd.DataFrame({'row_index': Xw.index, 'label': yw, 'score': scores}).to_csv(ROOT/'reports/lof_scores.csv', index=False)
result

**Interpretation.** LOF can find local pockets that a global detector misses.

**ML decision.** Compare it fairly using ranked scores rather than its default threshold.